# Задание
Сгенерируйте с использованием функции range (случайный шаг от 3 до 5) массив, содержащий отсортированные числа от 10 до 250 млн.

 

Можно использовать функцию randomint из модуля random для ещё большей рандомизации значений, но для целей работы алгоритма бинарного поиска проследите, чтобы значения в массиве были отсортированы.

 

Сгенерируйте с помощью list comprehensions и функции randomint (встроенный модуль random) 10 случайных чисел.

 

Напишите функцию для алгоритма линейного поиска.

 

Напишите функцию для алгоритма бинарного поиска.

 

Проверьте наличие ранее сгенерированных случайных чисел в массиве с помощью алгоритмов линейного и бинарного поиска, замерьте время

In [1]:
from csv import DictWriter
from pathlib import Path
from random import Random
from time import perf_counter_ns
from typing import Sequence


# --------------------------------------------------
# 1. Исходные параметры
# --------------------------------------------------

MIN_VALUE = 10
MAX_VALUE = 250_000_000
RANDOM_COUNT = 10

# Фиксированное значение позволяет повторить эксперимент
SEED = 20260917


# --------------------------------------------------
# 2. Функция линейного поиска
# --------------------------------------------------

def linear_search(numbers: Sequence[int], target: int) -> int:
    """
    Последовательно перебирает элементы.

    Возвращает индекс найденного элемента.
    Если элемент отсутствует, возвращает -1.
    """
    for index, value in enumerate(numbers):
        if value == target:
            return index

    return -1


# --------------------------------------------------
# 3. Функция бинарного поиска
# --------------------------------------------------

def binary_search(numbers: Sequence[int], target: int) -> int:
    """
    Ищет элемент в отсортированной последовательности,
    каждый раз уменьшая область поиска в два раза.

    Возвращает индекс найденного элемента.
    Если элемент отсутствует, возвращает -1.
    """
    left = 0
    right = len(numbers) - 1

    while left <= right:
        middle = (left + right) // 2
        value = numbers[middle]

        if value == target:
            return middle

        if value < target:
            left = middle + 1
        else:
            right = middle - 1

    return -1


# --------------------------------------------------
# 4. Генерация отсортированной последовательности
# --------------------------------------------------

rng = Random(SEED)

# Случайный шаг от 3 до 5 включительно
step = rng.randint(3, 5)

# Последовательность range уже отсортирована
numbers = range(
    MIN_VALUE,
    MAX_VALUE + 1,
    step
)


# --------------------------------------------------
# 5. Генерация 10 случайных чисел
# --------------------------------------------------

random_numbers = [
    rng.randint(MIN_VALUE, MAX_VALUE)
    for _ in range(RANDOM_COUNT)
]


# --------------------------------------------------
# 6. Поиск чисел и измерение времени
# --------------------------------------------------

results = []

for target in random_numbers:

    # Линейный поиск
    start_time = perf_counter_ns()
    linear_index = linear_search(numbers, target)
    linear_time_ns = perf_counter_ns() - start_time

    # Бинарный поиск
    start_time = perf_counter_ns()
    binary_index = binary_search(numbers, target)
    binary_time_ns = perf_counter_ns() - start_time

    # Проверяем, что алгоритмы дали одинаковый результат
    assert linear_index == binary_index

    results.append({
        "target": target,
        "found": linear_index != -1,
        "index": linear_index,
        "linear_time_ns": linear_time_ns,
        "linear_time_ms": linear_time_ns / 1_000_000,
        "binary_time_ns": binary_time_ns,
        "binary_time_ms": binary_time_ns / 1_000_000
    })


# --------------------------------------------------
# 7. Создание папки output
# --------------------------------------------------

output_dir = Path("output")

# Папка будет создана, если она ещё не существует
output_dir.mkdir(parents=True, exist_ok=True)

csv_path = output_dir / "search_results.csv"


# --------------------------------------------------
# 8. Сохранение результатов в CSV
# --------------------------------------------------

with csv_path.open(
    mode="w",
    newline="",
    encoding="utf-8-sig"
) as file:

    writer = DictWriter(
        file,
        fieldnames=results[0].keys()
    )

    writer.writeheader()
    writer.writerows(results)


# --------------------------------------------------
# 9. Вывод результатов
# --------------------------------------------------

print(f"Случайный шаг: {step}")
print(f"Количество элементов: {len(numbers):,}")
print(f"Первый элемент: {numbers[0]:,}")
print(f"Последний элемент: {numbers[-1]:,}")
print(f"Случайные числа: {random_numbers}")
print()

print(
    f"{'Число':>15} "
    f"{'Найдено':>10} "
    f"{'Индекс':>15} "
    f"{'Линейный, мс':>18} "
    f"{'Бинарный, мс':>18}"
)

for result in results:
    found_text = "да" if result["found"] else "нет"

    print(
        f"{result['target']:>15,} "
        f"{found_text:>10} "
        f"{result['index']:>15,} "
        f"{result['linear_time_ms']:>18.3f} "
        f"{result['binary_time_ms']:>18.6f}"
    )


# --------------------------------------------------
# 10. Суммарное время
# --------------------------------------------------

total_linear_time = sum(
    result["linear_time_ns"]
    for result in results
)

total_binary_time = sum(
    result["binary_time_ns"]
    for result in results
)

print()
print(
    "Общее время линейного поиска: "
    f"{total_linear_time / 1_000_000_000:.6f} с"
)

print(
    "Общее время бинарного поиска: "
    f"{total_binary_time / 1_000_000:.6f} мс"
)

print()
print(f"Результаты сохранены в файл: {csv_path.resolve()}")

Случайный шаг: 5
Количество элементов: 49,999,999
Первый элемент: 10
Последний элемент: 250,000,000
Случайные числа: [35087093, 175792772, 202127950, 237766367, 83330586, 69855721, 149218320, 63447302, 1234457, 38383689]

          Число    Найдено          Индекс       Линейный, мс       Бинарный, мс
     35,087,093        нет              -1           2968.839           0.019400
    175,792,772        нет              -1           2798.505           0.012200
    202,127,950         да      40,425,588           2280.507           0.013300
    237,766,367        нет              -1           2766.782           0.012600
     83,330,586        нет              -1           2616.175           0.013500
     69,855,721        нет              -1           2741.702           0.011800
    149,218,320         да      29,843,662           1654.155           0.011600
     63,447,302        нет              -1           2764.399           0.012900
      1,234,457        нет              -1       